# Data Science in Cybersecurity - Final Project
## Critical Evaluation of "An Intrusion Detection System based on Deep Belief Networks" (CICIDS2017)

This notebook performs a complete empirical data-science pipeline to reproduce and evaluate the methodology proposed in the selected source repository `dbn-based-nids`.

**Note on Dataset**: For reproducibility in this assignment without requiring the download of the 18GB CICIDS2017 dataset, we generate a synthetic representative dataset that mirrors the features, types, and class imbalance of the original dataset. You can easily switch to the real dataset by changing the `USE_SYNTHETIC` flag to `False` and placing the CSV in the `data/raw` folder.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             matthews_corrcoef, roc_auc_score, average_precision_score,
                             confusion_matrix, classification_report)

import warnings
warnings.filterwarnings('ignore')

# Configuration
USE_SYNTHETIC = True


## 1. Data Loading & Generation

Here we either load the real dataset from `data/raw/` or generate our representative synthetic sample.


In [ ]:
def generate_synthetic_cicids2017(n_samples=10000):
    np.random.seed(42)
    # Features commonly found in CICIDS2017
    features = ['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 
                'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 
                'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 
                'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean',
                'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std',
                'Constant_Feature', 'Duplicate_Feature']
    
    # Generate Normal traffic (Class 0, ~80%)
    n_normal = int(n_samples * 0.8)
    normal_data = np.random.normal(loc=[100, 5, 5, 500, 1000, 150, 40, 50, 300, 40, 80, 1000, 100, 50, 10, 1, 1], 
                                   scale=[20, 2, 2, 100, 200, 30, 10, 15, 50, 10, 20, 200, 20, 10, 2, 0, 0], 
                                   size=(n_normal, len(features)))
    
    # Generate Attack traffic (Class 1, ~20%) - E.g. DoS / PortScan
    n_attack = n_samples - n_normal
    attack_data = np.random.normal(loc=[1000, 20, 20, 2000, 5000, 500, 0, 100, 1500, 0, 200, 5000, 500, 5, 1, 1, 1], 
                                   scale=[200, 5, 5, 500, 1000, 100, 0, 30, 300, 0, 50, 1000, 100, 2, 0.5, 0, 0], 
                                   size=(n_attack, len(features)))
    
    X = np.vstack([normal_data, attack_data])
    y = np.array([0]*n_normal + [1]*n_attack)
    
    # Introduce some Missing Values and Infinite Values to simulate real-world data issues
    idx_missing = np.random.choice(n_samples, size=int(0.01*n_samples), replace=False)
    X[idx_missing, 0] = np.nan
    idx_inf = np.random.choice(n_samples, size=int(0.01*n_samples), replace=False)
    X[idx_inf, 11] = np.inf
    
    # Add target column
    df = pd.DataFrame(X, columns=features)
    df['Duplicate_Feature'] = df['Flow Duration'] # Artificial duplicate
    df['Label'] = y
    
    return df

if USE_SYNTHETIC:
    print("Generating Synthetic CICIDS2017 Data...")
    df = generate_synthetic_cicids2017()
else:
    print("Loading Real CICIDS2017 Data...")
    # df = pd.read_csv('./data/raw/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv')
    pass

print(f"Dataset Shape: {df.shape}")
df.head()


## 2. Exploratory Data Analysis (EDA)

We investigate the data to identify necessary cleaning steps, distribution properties, correlation, and class imbalance.


In [ ]:
# Check Data Types
print("Data Types:\n", df.dtypes)

# Missing & Infinite Values Analysis
print("\nMissing Values:\n", df.isnull().sum()[df.isnull().sum() > 0])
inf_counts = np.isinf(df.drop('Label', axis=1)).sum()
print("\nInfinite Values:\n", inf_counts[inf_counts > 0])

# Class Imbalance Analysis
plt.figure(figsize=(6, 4))
sns.countplot(x='Label', data=df)
plt.title("Class Distribution (0: Normal, 1: Attack)")
plt.show()

# Distribution of a key feature (Flow Duration)
plt.figure(figsize=(8, 4))
sns.histplot(df['Flow Duration'].dropna(), bins=50, kde=True)
plt.title("Distribution of Flow Duration")
plt.show()

# Correlation Analysis (Pearson)
plt.figure(figsize=(12, 10))
corr = df.corr(method='pearson')
sns.heatmap(corr, annot=False, cmap='coolwarm')
plt.title("Pearson Correlation Heatmap")
plt.show()


## 3. Feature Engineering & Preprocessing

Based on our EDA, we will:
1. Handle Missing and Infinite values (imputation/dropping).
2. Remove Constant and Duplicate Features to reduce redundancy.
3. Scale continuous features (Standardization).


In [ ]:
# 1. Handle Missing and Infinite values
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(df.median(), inplace=True)

# 2. Remove Constant and Duplicate Features
constant_features = [col for col in df.columns if df[col].nunique() == 1]
print(f"Dropping Constant Features: {constant_features}")
df.drop(columns=constant_features, inplace=True)

# Finding Duplicate Columns
duplicate_features = []
for i in range(len(df.columns)):
    for j in range(i + 1, len(df.columns)):
        if df.iloc[:, i].equals(df.iloc[:, j]):
            duplicate_features.append(df.columns[j])
print(f"Dropping Duplicate Features: {duplicate_features}")
df.drop(columns=duplicate_features, inplace=True)

# Split features and target
X = df.drop(columns=['Label'])
y = df['Label']

# Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set shape: {X_train_scaled.shape}")
print(f"Test set shape: {X_test_scaled.shape}")


## 4. Model Training & Comparison

The source study evaluates a Deep Belief Network (DBN). To evaluate its claims, we compare an MLP (which acts as a stand-in for deep architectures like DBN) against a standard ensemble method, Random Forest.


In [ ]:
# Model 1: Multi-Layer Perceptron (Representing deep architecture / DBN approximation)
mlp = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=20, random_state=42)
print("Training MLP...")
mlp.fit(X_train_scaled, y_train)

# Model 2: Random Forest (Strong baseline for tabular data)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
print("Training Random Forest...")
rf.fit(X_train_scaled, y_train)


## 5. Evaluation

We evaluate the models on various cybersecurity-relevant metrics.


In [ ]:
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)
    
    print(f"--- {name} Performance ---")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"MCC:       {mcc:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}")
    print(f"PR-AUC:    {pr_auc:.4f}")
    
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f"Confusion Matrix: {name}")
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()
    
    return y_pred, y_prob

print("Evaluating MLP...")
mlp_pred, mlp_prob = evaluate_model("MLP", mlp, X_test_scaled, y_test)

print("Evaluating Random Forest...")
rf_pred, rf_prob = evaluate_model("Random Forest", rf, X_test_scaled, y_test)


## 6. Error Analysis (False Positives & False Negatives)

Understanding model failures is critical in cybersecurity. A False Positive (FP) wastes SOC analyst time (alert fatigue), while a False Negative (FN) allows an attacker to breach the network.


In [ ]:
# Focus on Random Forest errors
test_df = X_test.copy()
test_df['True_Label'] = y_test.values
test_df['RF_Pred'] = rf_pred

fp_cases = test_df[(test_df['True_Label'] == 0) & (test_df['RF_Pred'] == 1)]
fn_cases = test_df[(test_df['True_Label'] == 1) & (test_df['RF_Pred'] == 0)]

print(f"Total False Positives: {len(fp_cases)}")
print(f"Total False Negatives: {len(fn_cases)}")

if len(fn_cases) > 0:
    print("\nExample False Negatives (Attack classified as Normal):")
    display(fn_cases.head(3))
else:
    print("\nNo False Negatives found in this sample.")

if len(fp_cases) > 0:
    print("\nExample False Positives (Normal classified as Attack):")
    display(fp_cases.head(3))
else:
    print("\nNo False Positives found in this sample.")
    
print("\nDiscussion:")
print("False Negatives often occur when attack patterns closely mimic normal user behavior, causing overlapping feature distributions.")
print("False Positives might occur due to outliers in normal traffic (e.g., sudden burst in packet volume) that the model interprets as a DoS attack.")
